<a href="https://colab.research.google.com/github/asdiFlv3/PHAS0056_assignments/blob/main/mini_project/build_and_train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install dependencies
runtime resets each time so better save the install, but seems like most of them are already installed

cell separated for clearer output

In [ ]:
! pip install spacy[transformers] spacy-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled tra

In [ ]:
! pip install jsonlines

In [ ]:
! python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 47.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
! nvidia-smi

Tue Mar 17 12:54:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Load google drive and data here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = "/content/drive/MyDrive/sentiment_analysis_data/phase2_annotated_data/prodigy_annotated_snippets_consolidated_llms_20240625.jsonl"

### Parse the JSONL for both sentimental and relevance model

Looking into the data, each line has an "accept" field containing a list with one label, like ["NOT_ABOUT_IMMIGRATION"] or ["NEUTRAL_TOWARDS_IMMIGRATION"]. We need to train two separate models, so the parsing logic differs for each.

For the relevance model, collapse all sentiment labels into IMMIGRATION vs NOT_ABOUT_IMMIGRATION. For the sentiment model, only use the immigration-relevant snippets and keep the three sentiment classes.

In [ ]:
import json
import random

records = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Total records loaded: {len(records)}")

# map each record's label ----
# first see what labels exist in the dataset
all_labels = set()
for r in records:
    for label in r.get("accept", []):
        all_labels.add(label)
print(f"Labels found: {all_labels}")

# prepare relevance data
# everything that is not "NOT_ABOUT_IMMIGRATION" becomes IMMIGRATION
relevance_data = []
for r in records:
    text = r["text"]
    label = r["accept"][0]  # Each record has exactly one label
    if label == "NOT_ABOUT_IMMIGRATION":
        cats = {"IMMIGRATION": False, "NOT_IMMIGRATION": True}
    else:
        cats = {"IMMIGRATION": True, "NOT_IMMIGRATION": False}
    relevance_data.append((text, cats))

# prepare sentiment data
# only keep immigration-relevant snippets, map to three classes
SENTIMENT_MAP = {
    "POSITIVE_TOWARDS_IMMIGRATION": "POSITIVE",
    "NEUTRAL_TOWARDS_IMMIGRATION": "NEUTRAL",
    "NEGATIVE_TOWARDS_IMMIGRATION": "NEGATIVE",
}

sentiment_data = []
for r in records:
    label = r["accept"][0]
    if label in SENTIMENT_MAP:
        text = r["text"]
        mapped = SENTIMENT_MAP[label]
        # for multiclass textcat, all labels must be present, one True, rest False
        cats = {
            "POSITIVE": mapped == "POSITIVE",
            "NEUTRAL": mapped == "NEUTRAL",
            "NEGATIVE": mapped == "NEGATIVE",
        }
        sentiment_data.append((text, cats))

print(f"Relevance samples: {len(relevance_data)}")
print(f"Sentiment samples: {len(sentiment_data)}")

# check class balance for both relevance and sentiment datasets
from collections import Counter

rel_counts = Counter()
for _, cats in relevance_data:
    for k, v in cats.items():
        if v:
            rel_counts[k] += 1
print(f"Relevance class distribution: {rel_counts}")

sent_counts = Counter()
for _, cats in sentiment_data:
    for k, v in cats.items():
        if v:
            sent_counts[k] += 1
print(f"Sentiment class distribution: {sent_counts}")

Total records loaded: 12458
Labels found: {'NOT_ABOUT_IMMIGRATION', 'NEUTRAL_TOWARDS_IMMIGRATION', 'POSITIVE_TOWARDS_IMMIGRATION', 'NEGATIVE_TOWARDS_IMMIGRATION'}
Relevance samples: 12458
Sentiment samples: 7207
Relevance class distribution: Counter({'IMMIGRATION': 7207, 'NOT_IMMIGRATION': 5251})
Sentiment class distribution: Counter({'NEUTRAL': 3205, 'POSITIVE': 2483, 'NEGATIVE': 1519})


### Convert to spaCy's Binary Format
spacy's training pipeline expects .spacy binary files, not raw text. convert the data using DocBin:

In [ ]:
import spacy
from spacy.tokens import DocBin

def create_docbin(data, nlp, output_path):
    """
    Convert list of (text, cats_dict) tuples into a .spacy DocBin file.

    args:
        data: list of (text, {"LABEL": True/False, ...}) tuples
        nlp: a spaCy Language object (used to create Doc objects)
        output_path: where to save the .spacy file
    """
    db = DocBin()
    for text, cats in data:
        doc = nlp.make_doc(text)  # tokenises without running the full pipeline
        doc.cats = cats
        db.add(doc)
    db.to_disk(output_path)
    print(f"Saved {len(data)} docs to {output_path}")

# load a blank English model just for tokenisation
nlp = spacy.blank("en")

def three_way_split(data, train_ratio=0.7, dev_ratio=0.15, test_ratio=0.15):
    random.shuffle(data)
    n = len(data)
    train_end = int(n * train_ratio)
    dev_end = int(n * (train_ratio + dev_ratio))
    return data[:train_end], data[train_end:dev_end], data[dev_end:]

# relevance split
rel_train, rel_dev, rel_test = three_way_split(relevance_data)

create_docbin(rel_train, nlp, "rel_train.spacy")
create_docbin(rel_dev, nlp, "rel_dev.spacy")
create_docbin(rel_test, nlp, "rel_test.spacy")

print(f"Relevance — Train: {len(rel_train)}, Dev: {len(rel_dev)}, Test: {len(rel_test)}")

# sentiment split
sent_train, sent_dev, sent_test = three_way_split(sentiment_data)

create_docbin(sent_train, nlp, "sent_train.spacy")
create_docbin(sent_dev, nlp, "sent_dev.spacy")
create_docbin(sent_test, nlp, "sent_test.spacy")

print(f"Sentiment — Train: {len(sent_train)}, Dev: {len(sent_dev)}, Test: {len(sent_test)}")

Saved 8720 docs to rel_train.spacy
Saved 1869 docs to rel_dev.spacy
Saved 1869 docs to rel_test.spacy
Relevance — Train: 8720, Dev: 1869, Test: 1869
Saved 5044 docs to sent_train.spacy
Saved 1081 docs to sent_dev.spacy
Saved 1082 docs to sent_test.spacy
Sentiment — Train: 5044, Dev: 1081, Test: 1082


### Create the Training Config
spaCy uses a config file to define the entire training pipeline. We need to separate configs for the relevance model (2 classes) and sentiment model (3 classes). First create a config file set up for transformer-based text classification with GPU training for relevance.

### Hyperparameter reference

All training parameters are patched into the `.cfg` file via regex before training.

| Parameter | What it controls | Typical value |
|---|---|---|
| `max_epochs` | Full passes over training data. Set > 0 to use epoch-based training; set to `0` to use `max_steps` instead | `0` (step-based) or `10–30` |
| `max_steps` | Total gradient updates. Only used when `max_epochs = 0` | `10000–20000` |
| `patience` | Early stopping: halt if dev score doesn't improve for this many steps | `1000–2000` |
| `initial_rate` | Peak learning rate (reached after warmup) | `2e-5` – `5e-5` for transformers |
| `warmup_steps` | Steps to linearly ramp LR from 0 → `initial_rate` | `200–500` |
| `dropout` | Dropout probability during training | `0.1` |
| `batcher size` | Words per batch (spaCy batches by word count, not sample count) | `2000–4000` |

**Epoch-based vs step-based training:**
- Set `max_epochs = 10` (and `max_steps = 0`) to train for exactly 10 full passes — useful when you know dataset size.
- Set `max_epochs = 0` and `max_steps = 20000` to train for a fixed number of gradient updates regardless of dataset size — more predictable on variable-size datasets.
- `patience` acts as early stopping in both modes (measured in evaluation steps, set by `eval_frequency`).

Edit the config to point to the data files and set the correct labels.

In [ ]:
# Create a base config file for the relevance model if it doesn't exist
! python -m spacy init config /content/drive/MyDrive/immigrant_models/relevance_config.cfg\
    --lang en \
    --pipeline textcat \
    --optimize accuracy \
    --gpu

# Create a base config file for the relevance model if it doesn't exist
! python -m spacy init config /content/drive/MyDrive/immigrant_models/sentiment_config.cfg\
    --lang en \
    --pipeline textcat \
    --optimize accuracy \
    --gpu

ℹ Generated config template specific for your use case
- Language: en
- Pipeline: textcat
- Optimize for: accuracy
- Hardware: GPU
- Transformer: roberta-base
✔ Auto-filled config with all values
✔ Saved config
/content/drive/MyDrive/immigrant_models/relevance_config.cfg
You can now add your data and train your pipeline:
python -m spacy train relevance_config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: textcat
- Optimize for: accuracy
- Hardware: GPU
- Transformer: roberta-base
✔ Auto-filled config with all values
✔ Saved config
/content/drive/MyDrive/immigrant_models/sentiment_config.cfg
You can now add your data and train your pipeline:
python -m spacy train sentiment_config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
import re

def patch_config(path, patches: dict):
    """Apply regex substitutions to a spaCy config file."""
    text = open(path).read()
    for pattern, replacement in patches.items():
        text = re.sub(pattern, replacement, text)
    with open(path, "w") as f:
        f.write(text)

patch_config("/content/drive/MyDrive/immigrant_models/relevance_config.cfg", {
    # data paths
    r'train = null':       'train = "rel_train.spacy"',
    r'dev = null':         'dev = "rel_dev.spacy"',
    # epochs / steps — set max_epochs > 0 for epoch-based training,
    # or keep at 0 and use max_steps for step-based training
    r'max_epochs = \d+':   'max_epochs = 0',
    r'max_steps = \d+':    'max_steps = 20000',
    r'patience = \d+':     'patience = 1600',
    # learning rate schedule
    r'initial_rate = \S+': 'initial_rate = 5e-5',
    r'warmup_steps = \d+': 'warmup_steps = 250',
    # batch size (words per batch)
    r'(batcher[\s\S]*?size\s*=\s*)\d+': r'\g<1>3000',
    # dropout
    r'dropout = \S+':      'dropout = 0.1',
})

! python -m spacy init fill-config /content/drive/MyDrive/immigrant_models/relevance_config.cfg /content/drive/MyDrive/immigrant_models/relevance_config.cfg

⚠ Nothing to auto-fill: base config is already complete
✔ Saved config
/content/drive/MyDrive/immigrant_models/relevance_config.cfg
You can now add your data and train your pipeline:
python -m spacy train relevance_config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
# sentiment model is harder (3 classes, smaller dataset) so we use:
# - lower learning rate to avoid overshooting
# - more warmup steps for a gentler ramp
# - more patience to avoid stopping too early


patch_config("/content/drive/MyDrive/immigrant_models/sentiment_config.cfg", {
    # data paths
    r'train = null':       'train = "sent_train.spacy"',
    r'dev = null':         'dev = "sent_dev.spacy"',
    # epochs / steps — set max_epochs > 0 for epoch-based training,
    # or keep at 0 and use max_steps for step-based training
    r'max_epochs = \d+':   'max_epochs = 0',
    r'max_steps = \d+':    'max_steps = 20000',
    r'patience = \d+':     'patience = 2000',
    # learning rate schedule
    r'initial_rate = \S+': 'initial_rate = 2e-5',
    r'warmup_steps = \d+': 'warmup_steps = 500',
    # batch size (words per batch)
    r'(batcher[\s\S]*?size\s*=\s*)\d+': r'\g<1>3000',
    # dropout
    r'dropout = \S+':      'dropout = 0.1',
})

! python -m spacy init fill-config /content/drive/MyDrive/immigrant_models/sentiment_config.cfg /content/drive/MyDrive/immigrant_models/sentiment_config.cfg

⚠ Nothing to auto-fill: base config is already complete
✔ Saved config
/content/drive/MyDrive/immigrant_models/sentiment_config.cfg
You can now add your data and train your pipeline:
python -m spacy train sentiment_config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


### Train the model separately

In [ ]:
# train the sentiment model
! python -m spacy train /content/drive/MyDrive/immigrant_models/sentiment_config.cfg \
    --output /content/drive/MyDrive/immigration_models/sentiment_output \
    --gpu-id 0

✔ Created output directory:
/content/drive/MyDrive/immigration_models/sentiment_output
ℹ Saving to output directory:
/content/drive/MyDrive/immigration_models/sentiment_output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 158kB/s]
config.json: 100% 481/481 [00:00<00:00, 3.98MB/s]
vocab.json: 899kB [00:00, 2.18MB/s]
merges.txt: 456kB [00:00, 989kB/s] 
tokenizer.json: 1.36MB [00:00, 1.88MB/s]
2026-03-17 12:56:38.933714: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 12:56:38.953420: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0

In [ ]:
# train the relevance model
# ---- Train the relevance model ----
!python -m spacy train /content/drive/MyDrive/immigrant_models/relevance_config.cfg \
    --output /content/drive/MyDrive/immigration_models/relevance_output \
    --gpu-id 0

✔ Created output directory:
/content/drive/MyDrive/immigration_models/relevance_output
ℹ Saving to output directory:
/content/drive/MyDrive/immigration_models/relevance_output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
2026-03-17 13:40:29.236710: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 13:40:29.256877: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773754829.280769   17610 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773754829.288504   17610 cuda_blas.cc:

### Evaluate and check results
load best model and inspect performance

In [ ]:
import spacy

# load and test relevance model
nlp_rel = spacy.load("/content/drive/MyDrive/immigration_models/relevance_output/model-best")
nlp_sent = spacy.load("/content/drive/MyDrive/immigration_models/sentiment_output/model-best")

# test on a couple of examples written by me, relevant / irrelevant
test_texts = [
    "The government must reduce immigration to protect British jobs.",
    "We discussed the budget allocation for the new hospital wing.",
]

for text in test_texts:
    doc = nlp_rel(text)
    print(f"Text: {text[:80]}...")
    print(f"  Predictions: {doc.cats}")
    print()

Text: The government must reduce immigration to protect British jobs....
  Predictions: {'IMMIGRATION': 0.9436132907867432, 'NOT_IMMIGRATION': 0.05638672411441803}

Text: We discussed the budget allocation for the new hospital wing....
  Predictions: {'IMMIGRATION': 0.05144128575921059, 'NOT_IMMIGRATION': 0.9485586881637573}



In [ ]:
# sentiment model
# test on a couple of examples written by me, positive / negative / neutral
test_immigration_texts = [
    "Immigrants have enriched our culture and strengthened our economy.",
    "The number of illegal border crossings has reached crisis levels.",
    "The Home Office processed 5,000 visa applications last quarter.",
]

for text in test_immigration_texts:
    doc = nlp_sent(text)
    print(f"Text: {text[:80]}...")
    print(f"  Predictions: {doc.cats}")
    print()

Text: Immigrants have enriched our culture and strengthened our economy....
  Predictions: {'POSITIVE': 0.9999103546142578, 'NEUTRAL': 7.277957047335804e-05, 'NEGATIVE': 1.680828063399531e-05}

Text: The number of illegal border crossings has reached crisis levels....
  Predictions: {'POSITIVE': 2.2658246962237172e-05, 'NEUTRAL': 4.4324639020487666e-05, 'NEGATIVE': 0.9999330043792725}

Text: The Home Office processed 5,000 visa applications last quarter....
  Predictions: {'POSITIVE': 0.00011348601401550695, 'NEUTRAL': 0.9998806715011597, 'NEGATIVE': 5.847909960721154e-06}



In [ ]:
from sklearn.metrics import classification_report

def evaluate_model(model_path, test_spacy_path, labels):
    """
    Load a trained model, run it on the test set, and print F-scores.

    Parameters:
        model_path: path to the model-best folder
        test_spacy_path: path to the .spacy test file
        labels: list of label names the model predicts
    """
    nlp = spacy.load(model_path)

    # load the test DocBin back into Doc objects
    doc_bin = DocBin().from_disk(test_spacy_path)
    # need a blank vocab to reconstruct the docs
    nlp_blank = spacy.blank("en")
    test_docs = list(doc_bin.get_docs(nlp_blank.vocab))

    true_labels = []
    pred_labels = []

    for gold_doc in test_docs:
        # the true label is whichever has True in the stored cats
        true_label = max(gold_doc.cats, key=gold_doc.cats.get)

        # run the model on the same text to get predictions
        pred_doc = nlp(gold_doc.text)
        pred_label = max(pred_doc.cats, key=pred_doc.cats.get)

        true_labels.append(true_label)
        pred_labels.append(pred_label)

    print(classification_report(true_labels, pred_labels, target_names=labels))

In [ ]:
# evaluate relevance model ----
print("=== RELEVANCE MODEL ===")
evaluate_model(
    "/content/drive/MyDrive/immigration_models/relevance_output/model-best",
    "rel_test.spacy",
    labels=["IMMIGRATION", "NOT_IMMIGRATION"]
)

# evaluate sentiment model
print("=== SENTIMENT MODEL ===")
evaluate_model(
    "/content/drive/MyDrive/immigration_models/sentiment_output/model-best",
    "sent_test.spacy",
    labels=["POSITIVE", "NEUTRAL", "NEGATIVE"]
)

=== RELEVANCE MODEL ===
                 precision    recall  f1-score   support

    IMMIGRATION       0.91      0.95      0.93      1095
NOT_IMMIGRATION       0.93      0.86      0.89       774

       accuracy                           0.91      1869
      macro avg       0.92      0.91      0.91      1869
   weighted avg       0.92      0.91      0.91      1869

=== SENTIMENT MODEL ===
              precision    recall  f1-score   support

    POSITIVE       0.61      0.53      0.57       214
     NEUTRAL       0.64      0.75      0.69       472
    NEGATIVE       0.74      0.64      0.69       396

    accuracy                           0.67      1082
   macro avg       0.67      0.64      0.65      1082
weighted avg       0.67      0.67      0.67      1082

